In [ ]:
# Notebook: Update Piki Scores
# 
# Computes a Piki score (1-100) for every song in the catalog based on
# user ratings (likes, superlikes, dislikes).  The score is derived from
# regularised, bias-corrected ratings that are then ranked into percentile
# buckets.  After scoring, the piki_score and investible_songs tables are
# refreshed in the database.

import warnings
warnings.filterwarnings('ignore')
from requests import get
from requests.exceptions import RequestException
from contextlib import closing
from bs4 import BeautifulSoup
import json
import requests
import csv
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statistics
import random
%matplotlib inline

# Connect to the museiq database and expose `connection`, `engine`, and `DB`
%run db.py

In [ ]:
def get_ratings_all(start_date, end_date, superlike_weight=3):
    """
    Load all user ratings from the database.

    Rating scale:
    - Super-like (seq=2): superlike_weight points
    - Like      (seq=0): 1 point
    - Dislike:          -1 point

    Returns a DataFrame with columns: timestamp, user_id, song_id, rating,
    plus song metadata (title, artist, genre, etc.)
    """
    sql = """
        SELECT * FROM (
            -- Super-likes
            SELECT us.timestamp, us.user_id, song_id, %s as rating
            FROM user_songs us
            WHERE seq = 2
            AND   us.timestamp>='%s' and us.timestamp<'%s'
            UNION

            -- Regular likes
            SELECT us.timestamp, us.user_id, song_id, 1 as rating
            FROM user_songs us
            WHERE seq = 0
            AND   us.timestamp>='%s' and us.timestamp<'%s'
            UNION

            -- Dislikes
            SELECT timestamp, us.user_id, song_id, -1 as rating
            FROM unliked_songs us
            WHERE   us.timestamp>='%s' and us.timestamp<'%s'
        ) t
        INNER JOIN song_ids s ON t.song_id = s.song_id
        INNER JOIN catalog c ON t.song_id = c.id
        LEFT JOIN artist_genre ag ON c.artist_id = ag.artist_id
        WHERE piki_video_url IS NOT NULL
        ORDER BY t.timestamp ASC
    """%(superlike_weight, start_date, end_date, start_date, end_date, start_date, end_date)

    res = DB.fetch(sql)
    df = pd.DataFrame(res)

    print(f"Loaded {len(df):,} ratings")
    print(f"  Users: {df['user_id'].nunique():,}")
    print(f"  Songs: {df['song_id'].nunique():,}")
    return df

In [ ]:
# Load all ratings from 2021 onwards.
# Superlikes are weighted 10x to reflect stronger user intent.
ratings = get_ratings_all('2021-01-01', '2027-01-01', superlike_weight=10)

In [ ]:
def adjust_ratings(ratings, min_ratings_user, min_ratings_item, item_reg, learning_rate):
    """
    Correct for user-level rating bias (stringency).

    Steps:
    1. Compute a regularised average rating per song (item_rating_avg).
       Regularisation pulls sparse items towards a neutral prior of 0.5.
    2. Filter out songs with fewer than min_ratings_item ratings.
    3. For each user, measure 'stringency': how far their ratings are
       above/below the song averages.  Users who consistently rate lower
       than average have positive stringency (they are harsh raters).
    4. Filter out users with fewer than min_ratings_user ratings.
    5. Adjust each raw rating upward by (learning_rate * stringency) so
       that harsh raters' songs are not systematically undervalued.

    Returns:
        ratings_with_stringency  - adjusted ratings DataFrame
        item_rating_avg          - regularised per-song mean rating
        item_rating_count        - number of ratings per song
        df_users                 - per-user stringency values
    """
    # --- Step 1: per-song statistics ---
    item_rating_avg   = ratings[['user_id', 'song_id', 'rating']].groupby('song_id').mean()
    item_rating_count = ratings[['user_id', 'song_id', 'rating']].groupby('song_id').count()

    # Regularise: blend empirical mean with a neutral prior (0.5) weighted by item_reg
    item_rating_avg['rating'] = (
        item_rating_avg['rating'] * item_rating_count['rating'] + 0.5 * item_reg
    ) / (item_rating_count['rating'] + item_reg)

    # Keep only songs that meet the minimum rating threshold
    df_items = item_rating_avg[item_rating_count['user_id'] >= min_ratings_item]['rating'].rename('item_rating')

    # --- Step 2: attach song means to each rating row ---
    ratings_with_item_means = ratings.merge(df_items, how='inner', left_on='song_id', right_on='song_id')

    # Stringency = song's average rating minus this user's rating for that song.
    # Positive value means the user rated lower than average (harsh rater).
    ratings_with_item_means['stringency'] = (
        ratings_with_item_means['item_rating'] - ratings_with_item_means['rating']
    )

    # --- Step 3: per-user stringency ---
    user_stringency_avg = ratings_with_item_means[['user_id', 'stringency']].groupby('user_id').mean()
    user_stringency_cnt = ratings_with_item_means[['user_id', 'stringency']].groupby('user_id').count()

    # Keep only users with enough ratings to estimate stringency reliably
    df_users = user_stringency_avg[user_stringency_cnt['stringency'] >= min_ratings_user]

    # --- Step 4: apply bias correction ---
    ratings_with_stringency = ratings.merge(df_users, how='inner', left_on='user_id', right_on='user_id')
    ratings_with_stringency['rating'] = (
        ratings_with_stringency['rating'] + learning_rate * ratings_with_stringency['stringency']
    )
    ratings_with_stringency = ratings_with_stringency.drop('stringency', axis=1)
    ratings_with_stringency = ratings_with_stringency[ratings_with_stringency['rating'].notna()]
    ratings_with_stringency = ratings_with_stringency[ratings_with_stringency['song_id'].isin(df_items.index)]

    return ratings_with_stringency, item_rating_avg, item_rating_count, df_users

In [ ]:
# --- Database update helpers ---
# These functions sync rating counts and artist IDs into the piki_score table.

def get_genre_list():
    """Return all genres ranked by number of tagged artists."""
    sql = 'SELECT genre, COUNT(*) AS cnt FROM artist_genre GROUP BY genre ORDER BY cnt DESC'
    return DB.fetch(sql)

def delete_piki_score_table():
    """Clear all rows from piki_score before a full recompute."""
    DB.execute('DELETE FROM piki_score')

def update_dislikes():
    """Write the total dislike count for each song into piki_score."""
    sql = '''UPDATE piki_score
             JOIN (SELECT song_id, COUNT(*) AS num_dislikes
                   FROM unliked_songs GROUP BY song_id) AS l
               ON piki_score.song_id = l.song_id
             SET piki_score.num_dislikes = l.num_dislikes'''
    DB.execute(sql)

def update_likes():
    """Write the total like count (seq=0) for each song into piki_score."""
    sql = '''UPDATE piki_score
             JOIN (SELECT song_id, COUNT(*) AS num_likes
                   FROM user_songs WHERE seq=0 GROUP BY song_id) AS l
               ON piki_score.song_id = l.song_id
             SET piki_score.num_likes = l.num_likes'''
    DB.execute(sql)

def update_superlikes():
    """Write the total superlike count (seq=2) for each song into piki_score."""
    sql = '''UPDATE piki_score
             JOIN (SELECT song_id, COUNT(*) AS num_superlikes
                   FROM user_songs WHERE seq=2 GROUP BY song_id) AS s
               ON piki_score.song_id = s.song_id
             SET piki_score.num_superlikes = s.num_superlikes'''
    DB.execute(sql)

def update_nulls():
    """Replace NULL counts with 0 so arithmetic on counts is safe."""
    DB.execute('UPDATE piki_score SET num_likes=0      WHERE num_likes IS NULL')
    DB.execute('UPDATE piki_score SET num_superlikes=0 WHERE num_superlikes IS NULL')
    DB.execute('UPDATE piki_score SET num_dislikes=0   WHERE num_dislikes IS NULL')

def update_artists():
    """Populate the artist_id column in piki_score from the catalog table."""
    sql = '''UPDATE piki_score ps
             INNER JOIN catalog c ON ps.song_id = c.id
             SET ps.artist_id = c.artist_id'''
    DB.execute(sql)

In [ ]:
# --- Investible songs helpers ---
# investible_songs is a derived table listing songs users can invest in,
# along with a cost computed from their like/dislike ratio.

def delete_investible_songs():
    """Clear the investible_songs table before rebuilding it."""
    DB.execute('DELETE FROM investible_songs')

def insert_investible_songs_expectation():
    """
    Populate investible_songs with a Bayesian cost per song.

    Cost formula (clamped to [2, 8]):
      cost = 10 * (likes + superlikes + 5) / (likes + superlikes + dislikes + 10)

    The +5 / +10 are a Laplace smoothing prior (5 pseudo-likes out of 10
    pseudo-ratings) so that songs with very few ratings are not extreme outliers.
    cost_adj applies a small fixed offset to the raw cost.
    """
    sql = '''INSERT INTO investible_songs
             SELECT ps.song_id,
               GREATEST(LEAST(ROUND(
                 10*(IFNULL(num_likes,0)+IFNULL(num_superlikes,0)+5) /
                    (IFNULL(num_likes,0)+IFNULL(num_superlikes,0)+IFNULL(num_dislikes,0)+10)
               , 1), 8), 2) AS cost,
               GREATEST(LEAST(ROUND(
                 -0.45 + 10*(num_likes+num_superlikes+5) /
                             (num_likes+num_superlikes+num_dislikes+10)
               , 0), 8), 2) AS cost_adj
             FROM piki_score ps
             INNER JOIN song_ids s ON ps.song_id = s.song_id
             GROUP BY ps.song_id
             ORDER BY cost DESC'''
    DB.execute(sql)

In [ ]:
# --- Iterative bias correction ---
# Run adjust_ratings repeatedly until the scores converge.
# Each iteration refines the stringency estimates using the already-adjusted
# ratings from the previous round.  15 iterations is typically sufficient.

item_reg         = 0   # no regularisation (all songs treated equally)
learning_rate    = 1   # full correction applied each step
min_ratings_user = 0   # include all users
min_ratings_item = 0   # include all songs

for i in range(15):
    prev_ratings = ratings.copy(deep=True)
    ratings, item_rating_avg, item_rating_count, df_users = adjust_ratings(
        ratings, min_ratings_user, min_ratings_item, item_reg, learning_rate
    )
    print(f"Iteration {i+1}: {len(ratings):,} ratings")

In [ ]:
# Confirm convergence: the maximum change in any single rating between the
# last two iterations should be near zero.
max_delta = ((ratings['rating'] - prev_ratings['rating']) ** 2).max()
print(f"Max squared rating change: {max_delta:.2e}  (should be ~0 if converged)")

In [ ]:
# --- Piki score bucketing ---
# Songs are split into groups by number of ratings, then scored within each
# group using percentile rank (qcut).  Groups with more ratings get a higher
# score ceiling, rewarding well-tested songs.
#
# ratings_list:     lower/upper bounds (number of ratings) for each bucket
# max_rating_list:  maximum Piki score allowed within that bucket

ratings_list     = [0,   2,  5, 10, 20,  50, 200, 10000]
max_rating_list  = [94, 95, 96, 97, 98,  99, 100]

In [ ]:
# Clear the piki_score table so we can rebuild it from scratch.
delete_piki_score_table()

# Assign a Piki score to every song by ranking it within its rating-count bucket.
for i in range(7):
    lower_bound = ratings_list[i]
    upper_bound = ratings_list[i + 1]
    print(f"Bucket {i+1}: {lower_bound}–{upper_bound} ratings, max score = {max_rating_list[i]}")

    # Select songs whose rating count falls within this bucket
    in_bucket = (item_rating_count['rating'] < upper_bound) & (item_rating_count['rating'] >= lower_bound)
    df_bucket = item_rating_avg[in_bucket]

    # Divide bucket into up to 100 percentile bands and label each band
    # with a score between (max_rating - 99) and max_rating.
    quantile_cuts = pd.qcut(df_bucket.rating, np.linspace(0, 1, 101), duplicates='drop')
    num_bands = len(quantile_cuts.cat.categories)
    score_floor = max_rating_list[i] - num_bands  # e.g. 100 - 85 = 15 → scores 16..100
    score_labels = score_floor + np.linspace(1, num_bands, num_bands)

    df_bucket = df_bucket.copy()
    df_bucket['piki_score'] = pd.qcut(
        df_bucket.rating, np.linspace(0, 1, 101), labels=score_labels, duplicates='drop'
    )

    # Write this bucket's scores to the piki_score table
    df_bucket.reset_index()[['song_id', 'piki_score']].to_sql(
        'piki_score', con=engine, if_exists='append', index=False
    )

In [ ]:
# Refresh raw like/dislike/superlike counts in piki_score from the source tables.
update_dislikes()
update_likes()
update_superlikes()
update_nulls()   # replace any NULLs introduced for songs with no activity

In [ ]:
# Populate the artist_id column (needed for artist-level aggregations).
update_artists()

In [ ]:
# Rebuild the investible_songs table with the freshly computed scores.
delete_investible_songs()
insert_investible_songs_expectation()
print("Done — piki_score and investible_songs tables updated.")